In [12]:
import re
import pandas as pd

from nltk.tokenize import sent_tokenize

import spacy
# spacy.cli.download('en_core_web_lg')
nlp = spacy.load("en_core_web_lg") 

In [ ]:
def cleaner(texto): # Limpa quebras de linha e barras
  cleaner = r'\\n|\\n\\n|\\'
  texto = re.sub(cleaner, ' ', texto)
  texto = re.sub('  ', ' ', texto)
  #texto = re.findall('\w+[!?:,\.]*', texto)
  return texto

In [ ]:
def feats(df):
  df['text'] = df['text'].apply(lambda txt: cleaner(txt))
  df['sent_tokens'] = df['text'].apply(lambda txt: sent_tokenize(txt))
  df['doc'] = df['text'].apply(lambda txt: nlp(txt))

In [ ]:
def n_grammer(doc): # Gera unigramas, bigramas e trigramas para o uso do MOL
    sentence = []
    for token in doc:
        if token.is_alpha:
            sentence.append(token.text)

    bi_result = []
    for word in range(len(sentence)-1):
        firstWord = sentence[word]
        secondWord = sentence[word+1]
        element = [firstWord.lower(), secondWord.lower()]
        bi_result.append(element)

    bigram_list = []
    for b_gram in bi_result:
        bigram = ' '.join(b_gram)
        bigram_list.append(bigram)

    tri_result = []
    for word in range(len(sentence)-2):
        firstWord = sentence[word]
        secondWord = sentence[word+1]
        thirdWord = sentence[word+2]
        element = [firstWord.lower(), secondWord.lower(), thirdWord.lower()]
        tri_result.append(element)

    trigram_list =[]
    for t_gram in tri_result:
        trigram = ' '.join(t_gram)
        trigram_list.append(trigram)
    
    unigram_list = sentence
    ngrams = [unigram_list, bigram_list, trigram_list]
    return ngrams

In [16]:
mol_terms = [w.rstrip() for w in open(r"..\data\raw\mol_terms.txt", 'r')]
mol_expressions = [w.rstrip() for w in open(r"..\data\raw\mol_expressions.txt", 'r')]

In [ ]:
def hate_speech_detector(ngram):
    unigrams = ngram[0]
    bgrams = ngram[1]
    tgrams = ngram[2]
    tot_hate_terms = 0
    tot_hate_expressions = 0

    total_hate_jj = 0 # Total de adjetivos de ódio
    total_hate_nn = 0 # Total de substantivos de ódio
    total_hate_rb = 0 # Total de advérbios de ódio
    total_hate_vb = 0 # Total de verbos de ódio

    for term in mol_terms:
        if term in unigrams:
            tot_hate_terms += 1
            tag = nlp(term)[0].tag_
            if tag.startswith('JJ'): total_hate_jj += 1
            elif tag.startswith('NN'): total_hate_nn += 1
            elif tag.startswith('RB'): total_hate_rb += 1
            elif tag.startswith('VB'): total_hate_vb += 1

    for bigram in bgrams:
        if bigram in mol_terms: tot_hate_terms += 1
        if bigram in mol_expressions: tot_hate_expressions += 1
    for tigram in tgrams:
        if tigram in mol_terms: tot_hate_terms += 1
        if tigram in mol_expressions: tot_hate_expressions += 1
    return pd.Series([tot_hate_terms, tot_hate_expressions, total_hate_jj, total_hate_nn, total_hate_rb, total_hate_vb])

In [ ]:
def preprocess(name=str, file_path=str):
    df_name = name
    raw_df = pd.read_json(file_path)
    df = raw_df[['id', 'text', 'labels']]

    df['persuasion'] = df['labels'].apply(lambda label: 1 if label else 0)

    df['text'] = df['text'].apply(cleaner)

    feats(df)

    df['ngrams'] = df['doc'].apply(n_grammer)

    df[['hate_speech_terms_total', 'hate_speech_expressions_total', 'total_hate_jj', 'total_hate_nn', 'total_hate_rb', 'total_hate_vb']] = df['ngrams'].apply(hate_speech_detector)
    
    print(df.head())
    
    return df

In [19]:
train = preprocess('train', r'..\data\raw\train.json')

      id                                               text  \
0  65635  THIS IS WHY YOU NEED A SHARPIE WITH YOU AT ALL...   
1  67927  GOOD NEWS! NAZANIN ZAGHARI-RATCLIFFE AND ANOOS...   
2  68031                            PAING PHYO MIN IS FREE!   
3  77490  Move your ships away! oooook Move your ships a...   
4  67641           WHEN YOU'RE THE FBI, THEY LET YOU DO IT.   

                                              labels  persuasion  \
0             [Black-and-white Fallacy/Dictatorship]           1   
1  [Loaded Language, Glittering generalities (Vir...           1   
2                                                 []           0   
3                                                 []           0   
4                       [Thought-terminating cliché]           1   

                                         sent_tokens  \
0  [THIS IS WHY YOU NEED A SHARPIE WITH YOU AT AL...   
1  [GOOD NEWS!, NAZANIN ZAGHARI-RATCLIFFE AND ANO...   
2                          [PAING PHYO MIN I

In [20]:
test = preprocess('test', r'..\data\raw\dev_subtask1_en.json')

      id                                               text  \
0  63292     This is why we're free This is why we're safe    
1  70419  IF YOU SAY WE'RE IN THE MIDDLE OF A DEADLY PAN...   
2  63673  AFTER A YEAR OF ZERO EVIDENCE OF RUSSIAN COLLU...   
3  71297                                      MY WAY MY WAY   
4  66340  Putin signed a decree to exclude Lyman from Ru...   

                                              labels  persuasion  \
0                        [Causal Oversimplification]           1   
1  [Loaded Language, Name calling/Labeling, Black...           1   
2  [Smears, Misrepresentation of Someone's Positi...           1   
3                                                 []           0   
4                                  [Loaded Language]           1   

                                         sent_tokens  \
0    [This is why we're free This is why we're safe]   
1  [IF YOU SAY WE'RE IN THE MIDDLE OF A DEADLY PA...   
2  [AFTER A YEAR OF ZERO EVIDENCE OF RUSSIAN

In [21]:
train.to_pickle(r"..\data\processed\train_cl.pkl")
test.to_pickle(r"..\data\processed\test_cl.pkl")